In [1]:

import numpy as np

# Kroupa (2001) slopes
alpha_nominal = np.array([0.3, 1.3, 2.3, 2.3])

# Published uncertainty ranges (not uniformly 1 sigma)
alpha_errors = np.array([0.7, 0.5, 0.3, 0.7])

# IMF boundaries, in solar masses
breaks = np.array([0.01, 0.08, 0.5, 1.0, np.inf])

# Ejection-fraction power-law exponent
alpha_ej = 0.27628889
s = 1 - alpha_ej


def power_integral(lower, upper, exponent):
    """Integral of x**exponent between two positive bounds."""
    p = exponent + 1

    if abs(p) < 1e-10:
        return np.log(upper / lower)

    return (upper**p - lower**p) / p


def imf_moment(s, m_min=0.01, m_max=100.0,
               alphas=alpha_nominal):
    """
    Normalized s-th moment of the Kroupa IMF.

    Stellar masses are expressed in solar masses.
    Returns a dimensionless quantity.
    """
    a0, a1, a2, a3 = alphas

    # Enforce continuity at each IMF breakpoint
    coeffs = np.array([
        1.0,
        0.08**(a1 - a0),
        0.08**(a1 - a0) * 0.5**(a2 - a1),
        0.08**(a1 - a0) * 0.5**(a2 - a1)
            * 1.0**(a3 - a2)
    ])

    numerator = 0.0
    denominator = 0.0

    for i in range(4):
        lo = max(m_min, breaks[i])
        hi = min(m_max, breaks[i + 1])

        if lo >= hi:
            continue

        numerator += coeffs[i] * power_integral(
            lo, hi, s - alphas[i]
        )

        denominator += coeffs[i] * power_integral(
            lo, hi, -alphas[i]
        )

    if denominator <= 0:
        raise ValueError("Invalid stellar mass range")

    return numerator / denominator


def moment_gradient(s, m_min=0.01, m_max=100.0,
                    step=1e-4):
    """
    Numerical partial derivatives of the IMF moment
    with respect to each Kroupa slope.
    """
    gradient = np.zeros(4)

    for i in range(4):
        upper = alpha_nominal.copy()
        lower = alpha_nominal.copy()

        upper[i] += step
        lower[i] -= step

        gradient[i] = (
            imf_moment(s, m_min, m_max, upper)
            - imf_moment(s, m_min, m_max, lower)
        ) / (2 * step)

    return gradient


def propagate_imf_error(s, m_min=0.01, m_max=100.0):
    """
    First-order propagation using the published
    slope uncertainty ranges as perturbation scales.

    This is NOT a calibrated 1-sigma error.
    Assumes independent slope perturbations.
    """
    moment = imf_moment(s, m_min, m_max)
    gradient = moment_gradient(s, m_min, m_max)

    contributions = np.abs(gradient) * alpha_errors
    total = np.sqrt(np.sum(contributions**2))

    return moment, total, contributions


# Example
moment, uncertainty_scale, contributions = (
    propagate_imf_error(s, m_min=0.01, m_max=100)
)

print(f"Moment: {moment:.6f}")
print(f"Linearized sensitivity scale: {uncertainty_scale:.6f}")
print(f"Relative sensitivity: {uncertainty_scale/moment:.2%}")

for i, contribution in enumerate(contributions):
    print(f"alpha_{i} contribution: {contribution:.6f}")


Moment: 0.383290
Linearized sensitivity scale: 0.183982
Relative sensitivity: 48.00%
alpha_0 contribution: 0.064369
alpha_1 contribution: 0.128480
alpha_2 contribution: 0.025340
alpha_3 contribution: 0.112057
